# `player_features` -- Player-Level Rolling Stats & Matchup History

**Ported from `project_hrishav/feature_engineering.py`** (`compute_rolling_player_stats`, `compute_matchup_features`), adapted from Cricsheet's JSON column schema to project_gagan's `ipl_data.xlsx` schema.

**This is additive/optional** -- it does *not* feed into `pipeline.py`'s team-level Elo/form/H2H features. It exists as a standalone, tested capability (player-level granularity is hrishav's core innovation) that a future model could opt into, per the approved merge decision not to force team-level and player-level features into one shared vector.

**Column mapping vs. the Cricsheet original:**

| Cricsheet (hrishav) | ipl_data.xlsx (this port) |
|---|---|
| `batter_runs` | `runs_batter` |
| `total_runs` | `runs_total` |
| `extra_runs` | `runs_extras` |
| `is_wide` | `extras_wides > 0` |
| `season_year` | `year` |
| `phase` | `(over > 6) + (over > 15)` (same formula as `src/features.phase`) |

**Difference from the original:** no pickle-caching layer. hrishav's `run_all.py` recomputed these stats on every process launch, so caching to disk made sense there; this project's pipeline doesn't call this module repeatedly in the same way, so the caching machinery was left out entirely rather than ported unused.

In [1]:
import pandas as pd

## `BOWLER_WICKET_KINDS`

Not every dismissal is credited to the bowler -- a run-out, for instance, is a fielding play, not a bowling achievement. This set defines exactly which `wicket_kind` values count toward a bowler's wicket tally: `caught`, `bowled`, `lbw`, `stumped`, `caught and bowled`, and `hit wicket`. Verified against the real `ipl_data.xlsx` data (see `wicket_kind` value counts inspected during the port) to confirm these are exactly the strings that appear in the dataset.

In [2]:
BOWLER_WICKET_KINDS = {"caught", "bowled", "lbw", "stumped", "caught and bowled", "hit wicket"}

## `_add_helper_columns()`

A small private helper shared by both public functions below: adds `is_wide` (boolean, derived from `extras_wides > 0`) and `phase` (0/1/2, same Powerplay/Middle/Death split used everywhere else in this project) to a copy of the input DataFrame, so the aggregation functions below don't have to repeat this logic.

In [3]:
def _add_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["is_wide"] = df["extras_wides"] > 0
    df["phase"] = (df["over"] > 6).astype(int) + (df["over"] > 15).astype(int)
    return df

## `compute_rolling_player_stats()` -- the public, no-lookahead entry point

**This is the function a caller actually uses.** It walks through IPL seasons in order and, for each year `Y`, computes every player's career batting/bowling stats *using only balls bowled in years strictly before `Y`* -- so a player's 2015 "rolling stats" reflect 2008-2014 only, never anything from 2015 itself. This mirrors exactly the same walk-forward principle as `src/features.py`'s `compute_form_h2h`, just at player granularity instead of team granularity.

The very first season in the dataset (2008) has **no prior history at all**, so it returns an empty dict for every player -- there's nothing to compute a rolling average from yet.

Implementation: iterates through sorted years, and before adding each year's data to a running `cumulative` DataFrame, calls `_aggregate_player_stats` on whatever has been accumulated *so far* (i.e. strictly prior years) to produce that year's lookup table.

In [4]:
def compute_rolling_player_stats(df: pd.DataFrame) -> dict[int, dict[str, dict]]:
    """
    Returns {year: {player_name: {bat_runs, bat_sr, ..., bowl_econ, ...}}}.

    For year Y, stats are derived from all balls with year < Y. The first
    year has all-zero features (no history available).
    """
    df = _add_helper_columns(df)
    years_sorted = sorted(df["year"].unique())
    season_stats: dict[int, dict] = {}
    cumulative = pd.DataFrame()

    for year in years_sorted:
        season_stats[year] = {} if cumulative.empty else _aggregate_player_stats(cumulative)
        cumulative = pd.concat([cumulative, df[df["year"] == year]], ignore_index=True)

    return season_stats

## `_aggregate_player_stats()` -- the actual stat computations

Given a DataFrame slice (all balls from prior seasons), computes every player's batting AND bowling stats. This is a private helper -- callers should always go through `compute_rolling_player_stats` above, which handles the walk-forward slicing correctly.

**Batting stats computed** (11 fields per player):
- `bat_innings`: distinct (match, innings) appearances
- `bat_runs` / `bat_balls`: career totals (wides excluded -- a   wide isn't a ball "faced" by the batter)
- `bat_avg`: `runs / dismissals` (falls back to raw runs if the   player has never been out, to avoid a division by zero -- an   imperfect convention shared with the hrishav original, since   a true "average" is undefined for a never-out player)
- `bat_sr`: strike rate, `runs/balls*100`
- `bat_4s` / `bat_6s` / `bat_boundary_rt`: boundary counts and rate
- `bat_pp_sr` / `bat_mid_sr` / `bat_death_sr`: strike rate broken   out by match phase, via the private `_phase_sr` closure

**Bowling stats computed** (10 fields per player):
- `bowl_balls` / `bowl_runs`: career totals (again, wides excluded)
- `bowl_wkts`: only wickets in `BOWLER_WICKET_KINDS` count
- `bowl_econ`: economy rate, `runs/balls*6`
- `bowl_avg` / `bowl_sr`: runs-per-wicket and balls-per-wicket
- `bowl_dot_rt`: fraction of deliveries that were dot balls
- `bowl_pp_econ` / `bowl_mid_econ` / `bowl_dth_econ`: economy by phase

Note the `stats.setdefault(player, {}).update({...})` pattern used for both batting and bowling: a player who has both batted and bowled ends up with one combined dict containing all 21 fields, rather than two separate entries.

In [5]:
def _aggregate_player_stats(df: pd.DataFrame) -> dict[str, dict]:
    stats: dict[str, dict] = {}

    # ---- Batting stats ----
    bat_legal = df[~df["is_wide"]]
    bat_g = bat_legal.groupby("batter")

    bat_runs = bat_g["runs_batter"].sum()
    bat_balls = bat_g["runs_batter"].count()
    bat_inns = bat_legal.groupby(["batter", "match_id"])["innings"].nunique().groupby("batter").count()
    bat_4s = (bat_legal["runs_batter"] == 4).groupby(bat_legal["batter"]).sum()
    bat_6s = (bat_legal["runs_batter"] == 6).groupby(bat_legal["batter"]).sum()
    bat_outs = df[df["is_wicket"] == 1].groupby("player_out").size()

    def _phase_sr(phase_id: int) -> pd.Series:
        p = bat_legal[bat_legal["phase"] == phase_id]
        r = p.groupby("batter")["runs_batter"].sum()
        b = p.groupby("batter")["runs_batter"].count()
        return (r / b * 100).fillna(0.0)

    pp_sr, mid_sr, dth_sr = _phase_sr(0), _phase_sr(1), _phase_sr(2)

    all_batters = bat_runs.index.union(bat_outs.index)
    for player in all_batters:
        r = bat_runs.get(player, 0)
        b = bat_balls.get(player, 0)
        inn = bat_inns.get(player, 0)
        outs = bat_outs.get(player, 0)

        stats.setdefault(player, {}).update({
            "bat_innings": int(inn),
            "bat_runs": int(r),
            "bat_balls": int(b),
            "bat_avg": round(r / outs, 4) if outs > 0 else float(r),
            "bat_sr": round(r / b * 100, 4) if b > 0 else 0.0,
            "bat_4s": int(bat_4s.get(player, 0)),
            "bat_6s": int(bat_6s.get(player, 0)),
            "bat_boundary_rt": round((bat_4s.get(player, 0) + bat_6s.get(player, 0)) / b, 4) if b > 0 else 0.0,
            "bat_pp_sr": round(pp_sr.get(player, 0.0), 4),
            "bat_mid_sr": round(mid_sr.get(player, 0.0), 4),
            "bat_death_sr": round(dth_sr.get(player, 0.0), 4),
        })

    # ---- Bowling stats ----
    bowl_legal = df[~df["is_wide"]]
    bowler_wk_mask = (df["is_wicket"] == 1) & df["wicket_kind"].isin(BOWLER_WICKET_KINDS)
    bowl_wk = df[bowler_wk_mask].groupby("bowler").size()

    bowl_g = bowl_legal.groupby("bowler")
    bowl_runs = bowl_g["runs_total"].sum()
    bowl_balls = bowl_g["runs_total"].count()
    bowl_dots = (bowl_legal["runs_batter"] == 0).groupby(bowl_legal["bowler"]).sum()

    def _bowl_phase_econ(phase_id: int) -> pd.Series:
        p = bowl_legal[bowl_legal["phase"] == phase_id]
        r = p.groupby("bowler")["runs_total"].sum()
        b = p.groupby("bowler")["runs_total"].count()
        return (r / b * 6).fillna(0.0)

    pp_econ, mid_econ, dth_econ = _bowl_phase_econ(0), _bowl_phase_econ(1), _bowl_phase_econ(2)

    for player in bowl_runs.index:
        r = bowl_runs.get(player, 0)
        b = bowl_balls.get(player, 0)
        wk = bowl_wk.get(player, 0)
        dots = bowl_dots.get(player, 0)

        stats.setdefault(player, {}).update({
            "bowl_balls": int(b),
            "bowl_runs": int(r),
            "bowl_wkts": int(wk),
            "bowl_econ": round(r / b * 6, 4) if b > 0 else 0.0,
            "bowl_avg": round(r / wk, 4) if wk > 0 else (float(r) if r else 0.0),
            "bowl_sr": round(b / wk, 4) if wk > 0 else 0.0,
            "bowl_dot_rt": round(dots / b, 4) if b > 0 else 0.0,
            "bowl_pp_econ": round(pp_econ.get(player, 0.0), 4),
            "bowl_mid_econ": round(mid_econ.get(player, 0.0), 4),
            "bowl_dth_econ": round(dth_econ.get(player, 0.0), 4),
        })

    return stats

## `compute_matchup_features()` -- batter-vs-bowler head-to-head history

The public, walk-forward entry point for **matchup-level** history (as opposed to individual player stats above): for every `(batter, bowler)` pair that has ever faced off, how did that specific matchup go historically? This captures something individual player stats can't -- a batter might dominate most bowlers but historically struggle against one specific bowling style/bowler.

Same walk-forward pattern as `compute_rolling_player_stats`: iterate through years in order, and for year `Y` only aggregate over strictly prior years.

In [6]:
def compute_matchup_features(df: pd.DataFrame) -> dict[int, dict[tuple, dict]]:
    """
    For each year, head-to-head batter-vs-bowler stats from prior years.

    Returns {year: {(batter, bowler): {balls, runs, dismissals, boundary_rt, dot_rt, matchup_sr}}}
    """
    df = _add_helper_columns(df)
    years_sorted = sorted(df["year"].unique())
    matchup_stats: dict[int, dict] = {}
    cumulative = pd.DataFrame()

    for year in years_sorted:
        matchup_stats[year] = {} if cumulative.empty else _aggregate_matchup(cumulative)
        cumulative = pd.concat([cumulative, df[df["year"] == year]], ignore_index=True)

    return matchup_stats

## `_aggregate_matchup()` -- the actual (batter, bowler) aggregation

Groups every legal delivery by the `(batter, bowler)` pair and computes:
- `balls` / `runs`: how many legal balls this batter has faced   from this specific bowler, and runs scored off them
- `boundaries` / `dots`: counts of 4s/6s and dot balls in this matchup
- `dismissals`: how many times this bowler has specifically   dismissed this batter -- computed separately via a groupby on   `(player_out, bowler)` restricted to `BOWLER_WICKET_KINDS`,   then joined onto the main `combined` table (a player who's   faced a bowler but never been dismissed by them won't appear   in the `dism` groupby at all, hence the `.reindex(...).fillna(0)`)
- `boundary_rt` / `dot_rt` / `matchup_sr`: the same kinds of   rate statistics as the player-level function above, just   scoped to one specific matchup

The final return value converts the DataFrame into a plain dict keyed by `(batter, bowler)` tuples, since that's a more natural lookup structure for a caller than a MultiIndex DataFrame.

In [7]:
def _aggregate_matchup(df: pd.DataFrame) -> dict[tuple, dict]:
    legal = df[~df["is_wide"]].copy()
    legal["is_boundary"] = legal["runs_batter"].isin([4, 6]).astype(int)
    legal["is_dot"] = ((legal["runs_batter"] == 0) & (legal["runs_extras"] == 0)).astype(int)

    wk_df = df[(df["is_wicket"] == 1) & df["wicket_kind"].isin(BOWLER_WICKET_KINDS)].copy()

    g = legal.groupby(["batter", "bowler"])
    balls = g["runs_batter"].count().rename("balls")
    runs = g["runs_batter"].sum().rename("runs")
    bnd = g["is_boundary"].sum().rename("boundaries")
    dot = g["is_dot"].sum().rename("dots")
    dism = wk_df.groupby(["player_out", "bowler"]).size().rename("dismissals")
    dism.index.names = ["batter", "bowler"]

    combined = pd.concat([balls, runs, bnd, dot], axis=1).fillna(0)
    combined["dismissals"] = dism.reindex(combined.index).fillna(0).astype(int)
    combined["boundary_rt"] = (combined["boundaries"] / combined["balls"]).round(4)
    combined["dot_rt"] = (combined["dots"] / combined["balls"]).round(4)
    combined["matchup_sr"] = (combined["runs"] / combined["balls"] * 100).round(4)

    return {(batter, bowler): row.to_dict() for (batter, bowler), row in combined.iterrows()}